# **Desafio Prático Final: Plataforma "Vagas Tech"**

## **1. Setup do Ambiente e Conexão**

### **1.1 Instalando o .NET SDK na máquina do Colab**

In [1]:
print("Instalando o repositório de pacotes da Microsoft...")
!wget <https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb> -O packages-microsoft-prod.deb -q
!dpkg -i packages-microsoft-prod.deb -q
!rm packages-microsoft-prod.deb

print("Instalando o .NET 8 SDK... Por favor, aguarde.")
!apt-get update -y -q > /dev/null
!apt-get install -y dotnet-sdk-8.0 -q > /dev/null

print("\nVerificando a instalação do .NET...")
!dotnet --version

Instalando o repositório de pacotes da Microsoft...
/bin/bash: line 1: https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb: No such file or directory
dpkg: error: cannot access archive 'packages-microsoft-prod.deb': No such file or directory
rm: cannot remove 'packages-microsoft-prod.deb': No such file or directory
Instalando o .NET 8 SDK... Por favor, aguarde.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Verificando a instalação do .NET...
=8.0.130


### **1.2 Criando o projeto e instalando pacotes de SQL**

In [2]:
print("Criando o novo projeto de console C# 'VagasTechApp'...")
# Cria o projeto na pasta 'VagasTechApp'
!dotnet new console -n VagasTechApp

print("\nInstalando a biblioteca Microsoft.Data.Sqlite para execução de SQL...")
# Adiciona o driver oficial do SQLite para C# apontando explicitamente para o projeto
!dotnet add VagasTechApp/VagasTechApp.csproj package Microsoft.Data.Sqlite -v 8.0.11

Criando o novo projeto de console C# 'VagasTechApp'...
=========
Welcome to .NET 8.0!
---------------------
SDK Version: 8.0.130

----------------
Installed an ASP.NET Core HTTPS development certificate.
To trust the certificate, view the instructions: https://aka.ms/dotnet-https-linux

----------------
Write your first app: https://aka.ms/dotnet-hello-world
Find out what's new: https://aka.ms/dotnet-whats-new
Explore documentation: https://aka.ms/dotnet-docs
Report issues and find source on GitHub: https://github.com/dotnet/core
Use 'dotnet --help' to see available commands or visit: https://aka.ms/dotnet-cli
--------------------------------------------------------------------------------------
The template "Console App" was created successfully.

Processing post-creation actions...
Restoring /content/VagasTechApp/VagasTechApp.csproj:
  Determining projects to restore...
  Restored /content/VagasTechApp/VagasTechApp.csproj (in 1.19 sec).
Restore succeeded.



Instalando a bib

### **1.3 Criando o arquivo físico do banco de dados**

In [3]:
import sqlite3

try:
    print("Criando o arquivo físico do banco de dados 'vagas_tech.db'...")
    # Abre a conexão (se o arquivo não existir, o SQLite cria um arquivo .db vazio na raiz)
    conexao = sqlite3.connect('vagas_tech.db')
    print("✅ Banco 'vagas_tech.db' criado fisicamente na pasta de arquivos!")
    conexao.close()
except Exception as e:
    print(f"🚨 Erro ao criar o banco: {e}")

Criando o arquivo físico do banco de dados 'vagas_tech.db'...
✅ Banco 'vagas_tech.db' criado fisicamente na pasta de arquivos!


## **2. Desenvolvimento do CRUD**

### **2.1 Cria um novo arquivo chamado `MetodosCRUD.cs`**

In [ ]:
%%writefile VagasTechApp/MetodosCRUD.cs
using System;
using Microsoft.Data.Sqlite;

public static class MetodosCRUD
{
    // --- 1. CREATE (Cadastrar Vaga) ---


    // --- 2. CREATE (Cadastrar Candidata) ---


    public static void EnviarCandidatura(
      SqliteConnection conexao,
      int idCandidatura,
      int idVaga, 
      int idCandidata)
      {
        string consultaCandidaturaExistente = @"
          SELECT EXISTS (
            SELECT 1
            FROM CANDIDATURAS
            WHERE ID_VAGA = @idVaga
              AND ID_CANDIDATA = @idCandidata
          );
        ";

        using var comandoCandidaturaExistente = new SqliteCommand(consultaCandidaturaExistente, conexao);

        comandoCandidaturaExistente.Parameters.AddWithValue("@idVaga", idVaga);

        comandoCandidaturaExistente.Parameters.AddWithValue("@idCandidata", idCandidata);

        bool candidaturaExiste = Convert.ToBoolean(comandoCandidaturaExistente.ExecuteScalar());

        if (candidaturaExiste)
          throw new InvalidOperationException($"A candidata {idCandidata} já possui uma candidatura para a vaga {idVaga}.");

        string sql = @"
            INSERT INTO CANDIDATURAS
            (
                ID_CANDIDATURA,
                DATA_ENVIO,
                ID_VAGA,
                ID_CANDIDATA
            )
            VALUES
            (
                @idCandidatura,
                @dataEnvio,
                @idVaga,
                @idCandidata
            );
        ";

        try
        {
            using var comando = new SqliteCommand(sql, conexao);

            comando.Parameters.AddWithValue("@idCandidatura", idCandidatura);

            comando.Parameters.AddWithValue("@dataEnvio", DateTime.Now);

            comando.Parameters.AddWithValue("@idVaga", idVaga);

            comando.Parameters.AddWithValue("@idCandidata", idCandidata);

            comando.ExecuteNonQuery();

            Console.WriteLine($"Candidatura {idCandidatura} enviada com sucesso!");
        }
        catch (SqliteException ex) when (ex.SqliteErrorCode == 19)
        {
            throw new InvalidOperationException($"O ID da candidatura {idCandidatura} já existe.");
        }

      }


    // --- 5. UPDATE (Atualizar Salario Vaga) ---


    // --- 6. DELETE (Cancelar Candidatura) ---

}

Writing VagasTechApp/MetodosCRUD.cs


### **2.2 Compilando o projeto para verificar se há erros de sintaxe**

In [6]:
!dotnet build  VagasTechApp

=MSBuild version 17.8.49+7806cbf7b for .NET
  Determining projects to restore...
  All projects are up-to-date for restore.
  VagasTechApp -> /content/VagasTechApp/bin/Debug/net8.0/VagasTechApp.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:05.67


## **3. Simulação e Integração**

### **3.1 Lógica de integração no arquivo `Program.cs`**

In [ ]:
%%writefile VagasTechApp/Program.cs
using System;
using Microsoft.Data.Sqlite;

namespace VagasTechApp
{
    class Program
    {
        static void Main(string[] args)
        {
            var stringConexao = "Data Source=/content/vagas_tech.db";

            try
            {
                Console.WriteLine("=================================================");
                Console.WriteLine("INICIANDO INTEGRAÇÃO DA PLATAFORMA VAGAS TECH...");
                Console.WriteLine("=================================================\n");

                using (var conexao = new SqliteConnection(stringConexao))
                {
                    conexao.Open();
                    Console.WriteLine("🔋 Conexão estabelecida com o arquivo 'vagas_tech.db'!");

					// --- 1. CREATE (Cadastrar Vaga) ---


					// --- 2. CREATE (Cadastrar Candidata) ---


					// --- 4. READ (Consultar Candidaturas) ---


					// --- 5. UPDATE (Atualizar Salario Vaga) ---


					// --- 6. DELETE (Cancelar Candidatura) ---
                }
            }
            catch (Exception ex)
            {
                Console.WriteLine($"🚨 Ocorreu um erro na conexão: {ex.Message}");
            }
            finally
            {
                Console.WriteLine("=================================================");
                Console.WriteLine("PROCESSO DE INTEGRAÇÃO FINALIZADO.");
                Console.WriteLine("=================================================");
            }
        }
    }
}


Overwriting VagasTechApp/Program.cs
